In [ ]:
from __future__ import annotations

import os
import sys

sys.path.append(os.path.abspath("../"))


import os
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# Boxplots

In [ ]:
"""Bar-chart visualization of per-bin meter errors for the distance system."""

# ── Configuration ─────────────────────────────────────────────────────────────

SPEC_STR = "d_9-K_4-4-11"

DATA_DIR = Path("../../logs/fit/final/20260711-000000/matches/")

BINS = [0, 3, 5, 7, 9]
BIN_LABELS = ["0-3 m", "3-5 m", "5-7 m", "7-9 m"]

# d_max=3 omitted: produces only one bin, no meaningful trend
D_MAX_VALUES = [3, 5, 7, 9]

# Colorblind-safe palette (Wong 2011)
COLORS_SYSTEM: dict[int, str] = {
    3: "#E69F00",  # orange
    5: "#56B4E9",  # sky blue
    7: "#009E73",  # bluish green
    9: "#D55E00",  # vermillion
}
COLOR_BHUMAN = "#CC79A7"

BOX_WIDTH = 0.13
CLIP_AT = 13  # cm — boxes whose median is above this are clipped and annotated


# ── Data types ────────────────────────────────────────────────────────────────


@dataclass
class BinResult:
    errors: np.ndarray  # raw absolute errors in cm for this bin

    @property
    def n(self) -> int:
        return len(self.errors)

    @property
    def is_valid(self) -> bool:
        return self.n > 0


@dataclass
class Series:
    bins: list[np.ndarray]  # one array of errors per bin (empty array = invalid)
    color: str
    label: str

    @classmethod
    def from_bin_results(cls, bin_results: list[BinResult], color: str, label: str) -> Series:
        return cls(
            bins=[b.errors for b in bin_results],
            color=color,
            label=label,
        )

    def is_valid_at(self, bin_idx: int) -> bool:
        return len(self.bins[bin_idx]) > 0

    def first_valid_bin(self) -> int:
        return next(i for i in range(len(self.bins)) if self.is_valid_at(i))


# ── Data loading & processing ─────────────────────────────────────────────────


def load(path: Path) -> np.ndarray:
    """Load a .npy file with shape (N, 2): [d_pred, d_gt] in metres."""
    return np.load(path)


# def compute_errors(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
#     """Return (absolute_errors_cm, d_gt_m) from a (N, 2) array."""
#     d_pred, d_gt = data[:, 0], data[:, 1]
#     return np.abs(d_pred - d_gt) * 100, d_gt
def compute_errors(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (absolute_errors_cm, d_gt_m) from either:
    - (N, 2)    array where [:, 0] = d_pred (m), [:, 1] = d_gt (m)
    - (N, 2, 2) array where [:, 0, :] = pred (x,y), [:, 1, :] = gt (x,y)
                and distances are pixel Euclidean distances between each pair.
    """
    if data.ndim == 3 and data.shape[1:] == (2, 2):
        pred_coords, gt_coords = data[:, 0, :], data[:, 1, :]
        d_pred = np.linalg.norm(pred_coords, axis=-1)
        d_gt = np.linalg.norm(gt_coords, axis=-1)
        factor = 1
    elif data.ndim == 2 and data.shape[1] == 2:
        d_pred, d_gt = data[:, 0], data[:, 1]
        factor = 100
    else:
        raise ValueError(f"Expected shape (N, 2) or (N, 2, 2), got {data.shape}")

    return np.abs(d_pred - d_gt) * factor, d_gt


def bin_errors(
    errors: np.ndarray,
    d_gt: np.ndarray,
    d_max: float = np.inf,
) -> list[BinResult]:
    """
    Group raw errors into distance bins.

    Bins whose lower edge is >= d_max are marked invalid (empty array),
    since those distances were outside the system's operating range.
    """
    results: list[BinResult] = []
    for lo, hi in zip(BINS[:-1], BINS[1:], strict=False):
        if lo >= d_max:
            results.append(BinResult(np.array([])))
            continue
        mask = (d_gt >= lo) & (d_gt < hi)
        results.append(BinResult(errors[mask]))
    return results


# ── Drawing helpers ───────────────────────────────────────────────────────────


def _draw_box(
    ax: plt.Axes,
    xi: float,
    series: Series,
    bin_idx: int,
    *,
    add_label: bool,
) -> None:
    """Draw a single boxplot (possibly clipped) with sample-count annotation."""
    data = series.bins[bin_idx]
    color = series.color
    median = float(np.median(data))
    clipped = median > CLIP_AT

    # Clip whisker/box tops visually when the median itself is over the limit
    plot_data = np.clip(data, 0, CLIP_AT) if clipped else data

    bp = ax.boxplot(
        data,
        positions=[xi],
        widths=BOX_WIDTH,
        patch_artist=True,
        manage_ticks=False,
        # meanline=False,
        # showmeans=True,
        zorder=3,
        showfliers=True,
        medianprops=dict(color="black", linewidth=1.5),
        boxprops=dict(facecolor=color, alpha=0.85),
        whiskerprops=dict(color=color),
        capprops=dict(color=color),
        flierprops=dict(
            marker="o", markerfacecolor=color, markeredgecolor=color, markersize=3, alpha=0.5
        ),
        label=series.label if add_label else "_nolegend_",
    )

    if clipped:
        # Hatched top stripe to signal the box was cut off
        stripe_height = 1.0
        stripe_bottom = CLIP_AT - stripe_height

        ax.bar(
            xi,
            stripe_height,
            bottom=stripe_bottom,
            width=BOX_WIDTH - 0.01,
            color="white",
            edgecolor=color,
            linewidth=1.0,
            zorder=4,
        )
        ax.bar(
            xi,
            stripe_height,
            bottom=stripe_bottom,
            width=BOX_WIDTH - 0.01,
            color="none",
            edgecolor=color,
            linewidth=0.8,
            hatch="////",
            zorder=5,
        )

        q1, med, q3 = np.percentile(data, [25, 50, 75])
        ax.text(
            xi,
            stripe_bottom + (stripe_height / 2) - 0.1,
            f"Median={med:.0f} cm",
            ha="center",
            va="bottom",
            fontsize=8,
            color=color,
            fontweight="bold",
            zorder=7,
            bbox=dict(
                facecolor="white",
                edgecolor="lightgray",
                boxstyle="round,pad=0.2",
                alpha=0.8,
            ),
        )

    n = len(data)
    ax.text(
        xi + BOX_WIDTH / 5,
        -1.5,
        f"n={n}",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=50,
        color="black",
        zorder=5,
        bbox=dict(
            facecolor=color,
            edgecolor="lightgray",
            boxstyle="round,pad=0.2",
            alpha=0.65,
        ),
    )


# ── Main plot function ────────────────────────────────────────────────────────


def build_series(
    data_system: dict[int, np.ndarray],
    data_bhuman: np.ndarray,
    data_system_meter=None,
    data_bhuman_meter=None,
) -> list[Series]:
    """Construct all Series objects (B-Human first, then system variants)."""
    bh_errors, bh_dgt = compute_errors(data_bhuman)
    if data_bhuman_meter is not None:
        _, bh_dgt = compute_errors(data_bhuman_meter)
    series_list = [
        Series.from_bin_results(
            bin_errors(bh_errors, bh_dgt),
            color=COLOR_BHUMAN,
            label="B-Human",
        )
    ]
    for d_max in D_MAX_VALUES:
        errors, d_gt = compute_errors(data_system[d_max])
        if data_system_meter is not None:
            _, d_gt = compute_errors(data_system_meter[d_max])
        series_list.append(
            Series.from_bin_results(
                bin_errors(errors, d_gt, d_max=d_max),
                color=COLORS_SYSTEM[d_max],
                label=rf"$d_{{\mathrm{{max}}}} = {d_max}\,\mathrm{{m}}$",
            )
        )
    return series_list


def plot_meter_error(
    data_system: dict[int, np.ndarray],
    data_bhuman: np.ndarray,
    category_name: str,
    data_system_meter=None,
    data_bhuman_meter=None,
    save_path: Path | None = None,
    pixel_mode: bool = False,
) -> None:
    """Plot per-bin absolute meter errors for the system vs. B-Human baseline."""
    all_series = build_series(data_system, data_bhuman, data_system_meter, data_bhuman_meter)

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(BIN_LABELS))

    labeled: set[int] = set()

    for bin_idx in range(len(BIN_LABELS)):
        valid = [s for s in all_series if s.is_valid_at(bin_idx)]
        if not valid:
            continue

        n = len(valid)
        offsets = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * BOX_WIDTH

        for series, offset in zip(valid, offsets, strict=False):
            series_id = id(series)
            add_label = series_id not in labeled and bin_idx == series.first_valid_bin()
            if add_label:
                labeled.add(series_id)

            _draw_box(ax, x[bin_idx] + offset, series, bin_idx, add_label=add_label)

    ax.set_xticks(x)
    ax.set_xticklabels(BIN_LABELS)
    ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)

    if pixel_mode:
        ticks = list(range(0, 7, 1)) + list(range(8, 15, 2))
        ax.set_yticks(ticks)
        ax.set_ylim(-2, 15)
        ax.set_ylabel(r"$\bar{e}_{c,\mathrm{pixel}}^{\prime\prime}$ [px]")
    else:
        ticks = list(range(0, 7, 1)) + list(range(8, 13, 2))
        ax.set_yticks(ticks)
        ax.set_ylim(-2, 13)
        ax.set_ylabel(r"$\bar{e}_{c,\mathrm{metric}}^{\prime\prime}$ [cm]")

    ax.set_xlabel("Distanz [m]")
    # ax.legend(fontsize=9, loc="upper right" if pixel_mode else "upper left")
    ax.legend(
        fontsize=9,
        loc="upper left",
        bbox_to_anchor=(1.01, 1),
        borderaxespad=0,
    )

    plt.tight_layout()  # still call this first
    # then expand the right margin to fit the legend
    fig.subplots_adjust(right=0.78)  # tweak 0.78 to taste
    # ax.set_title(
    #     f"{title_label(object_name, n)} · Distanz {distance} m \nPrecision {avg_p:.3f}  ·  Recall {avg_r:.3f}",
    #     fontsize=12,
    #     fontweight="bold",
    #     pad=12,
    #     color="#1a1a2e",
    # )

    ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    plt.tight_layout()

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        fig.savefig(f"{save_path}/{category_name}.pdf", bbox_inches="tight")

    plt.show()

In [ ]:
# ── Usage: Ball ───────────────────────────────────────────────────────────────
data_system_ball = {
    d_max: load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "ball_1" / "model_distances.npy")
    for d_max in D_MAX_VALUES
}
data_bhuman_ball = load(DATA_DIR / "d_9-K_4-4-11" / "ball_1" / "bhuman_distances.npy")

plot_meter_error(
    data_system_ball,
    data_bhuman_ball,
    category_name="Ball",
    save_path=Path("../../plots/regression_error/meter/").as_posix(),
)

# ── Usage: Elfmeterpunkt ──────────────────────────────────────────────────────
data_system_ep = {
    d_max: load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "penaltyMark" / "model_distances.npy")
    for d_max in D_MAX_VALUES
}
data_bhuman_ep = load(DATA_DIR / "d_9-K_4-4-11" / "penaltyMark" / "bhuman_distances.npy")

plot_meter_error(
    data_system_ep,
    data_bhuman_ep,
    category_name="Elfmeterpunkt",
    save_path=Path("../../plots/regression_error/meter/").as_posix(),
)

# ── Usage: Linienkreuzungen (gepoolt über L/T/X) ──────────────────────────────
# Kreuzungstypen werden concatenated -> ein gepoolter Datensatz pro d_max,c
CROSSING_TYPES = ["L", "T", "X"]

data_system_inter = {
    d_max: np.concatenate(
        [
            load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "intersections" / t / "model_distances.npy")
            for t in CROSSING_TYPES
        ]
    )
    for d_max in D_MAX_VALUES
}
data_bhuman_inter = np.concatenate(
    [
        load(DATA_DIR / "d_9-K_4-4-11" / "intersections" / t / "bhuman_distances.npy")
        for t in CROSSING_TYPES
    ]
)

plot_meter_error(
    data_system_inter,
    data_bhuman_inter,
    category_name="Linienkreuzungen (gepoolt)",
    save_path=Path("../../plots/regression_error/meter/").as_posix(),
)

In [ ]:
# ── Usage: Ball ───────────────────────────────────────────────────────────────
data_system_ball_px = {
    d_max: load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "ball_1" / "model_matches.npy")
    for d_max in D_MAX_VALUES
}
data_bhuman_ball_px = load(DATA_DIR / "d_9-K_4-4-11" / "ball_1" / "bhuman_matches.npy")

data_system_ball_meter = {
    d_max: load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "ball_1" / "model_distances.npy")
    for d_max in D_MAX_VALUES
}
data_bhuman_ball_meter = load(DATA_DIR / "d_9-K_4-4-11" / "ball_1" / "bhuman_distances.npy")

plot_meter_error(
    data_system_ball_px,
    data_bhuman_ball_px,
    "Ball",
    data_system_ball_meter,
    data_bhuman_ball_meter,
    save_path=Path("../../plots/regression_error/pixel/").as_posix(),
    pixel_mode=True,
)

# ── Usage: Elfmeterpunkt ──────────────────────────────────────────────────────
data_system_ep = {
    d_max: load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "penaltyMark" / "model_distances.npy")
    for d_max in D_MAX_VALUES
}
data_bhuman_ep = load(DATA_DIR / "d_9-K_4-4-11" / "penaltyMark" / "bhuman_distances.npy")

data_system_ep_px = {
    d_max: load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "penaltyMark" / "model_matches.npy")
    for d_max in D_MAX_VALUES
}
data_bhuman_ep_px = load(DATA_DIR / "d_9-K_4-4-11" / "penaltyMark" / "bhuman_matches.npy")

plot_meter_error(
    data_system_ep_px,
    data_bhuman_ep_px,
    "Elfmeterpunkt",
    data_system_ep,
    data_bhuman_ep,
    save_path=Path("../../plots/regression_error/pixel/").as_posix(),
    pixel_mode=True,
)

# ── Usage: Linienkreuzungen (gepoolt über L/T/X) ──────────────────────────────
# Kreuzungstypen werden concatenated -> ein gepoolter Datensatz pro d_max,c
CROSSING_TYPES = ["L", "T", "X"]

data_system_inter = {
    d_max: np.concatenate(
        [
            load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "intersections" / t / "model_distances.npy")
            for t in CROSSING_TYPES
        ]
    )
    for d_max in D_MAX_VALUES
}
data_bhuman_inter = np.concatenate(
    [
        load(DATA_DIR / "d_9-K_4-4-11" / "intersections" / t / "bhuman_distances.npy")
        for t in CROSSING_TYPES
    ]
)

data_system_inter_px = {
    d_max: np.concatenate(
        [
            load(DATA_DIR / f"d_{d_max}-K_4-4-11" / "intersections" / t / "model_matches.npy")
            for t in CROSSING_TYPES
        ]
    )
    for d_max in D_MAX_VALUES
}
data_bhuman_inter_px = np.concatenate(
    [
        load(DATA_DIR / "d_9-K_4-4-11" / "intersections" / t / "bhuman_matches.npy")
        for t in CROSSING_TYPES
    ]
)

plot_meter_error(
    data_system_inter_px,
    data_bhuman_inter_px,
    "Linienkreuzungen (gepoolt)",
    data_system_inter,
    data_bhuman_inter,
    save_path=Path("../../plots/regression_error/pixel/").as_posix(),
    pixel_mode=True,
)